# 095 — Procedencia, marcas y autenticidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** σ = √(100·0.5·0.5) = 5. (a) z = (70 − 50)/5 = **4.0** → p ≈ 3 × 10⁻⁵,
se rechaza H₀: marca presente. (b) z = (55 − 50)/5 = **1.0** → totalmente compatible
con el azar: sin evidencia de marca (que no es lo mismo que "sin marca").

**Ejercicio 2.** Esperados bajo H₀: γT = 0.25 · 200 = **50** verdes.
σ = √(200 · 0.25 · 0.75) = √37.5 ≈ 6.124. Umbral: 50 + 4 · 6.124 ≈ 74.5 →
**k mínimo = 75** (con k = 75, z ≈ 4.08). Nota: γ más bajo concentra la señal —
cada token verde extra "vale más" desviaciones.

**Ejercicio 3.** z = (60 − 50)/5 = **2.0**: la evidencia pasa de contundente a débil.
El atacante solo necesita reescribir texto (cualquier paráfrasis rompe la correlación
token-anterior → lista verde), no la clave. C2PA no sufre dilución —la firma es válida
o no lo es— pero sufre el ataque análogo de **eliminación**: quitar el manifiesto deja
el activo sin procedencia, sin alarma alguna. Ambas garantías son unidireccionales.

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; la evidencia es lo único
que autoriza conclusiones.

In [ ]:
result = run_lab("safety", seed=95)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica de los ejercicios
import math

def z_score(k, T, gamma):
    return (k - gamma * T) / math.sqrt(T * gamma * (1 - gamma))

# Ejercicio 1
print(f"z(k=70, T=100, γ=0.5) = {z_score(70, 100, 0.5):.2f}")
print(f"z(k=55, T=100, γ=0.5) = {z_score(55, 100, 0.5):.2f}")
assert z_score(70, 100, 0.5) == 4.0

# Ejercicio 2: k mínimo con z ≥ 4 para γ=0.25, T=200
gamma, T = 0.25, 200
k_min = next(k for k in range(int(gamma * T), T + 1) if z_score(k, T, gamma) >= 4)
print(f"esperados bajo H0 = {gamma*T:.0f} · k mínimo con z ≥ 4: {k_min} "
      f"(z = {z_score(k_min, T, gamma):.2f})")
assert k_min == 75

# Ejercicio 3: dilución por paráfrasis
print(f"z(k=60, T=100, γ=0.5) = {z_score(60, 100, 0.5):.2f}  → evidencia débil")

## Reflexión

1. Con γ = 0.5, un 70 % de tokens verdes da z = 4.0 si T = 100, pero solo z ≈ 1.26 si T = 10. ¿Por qué la longitud del texto es decisiva para el detector y qué implica para fragmentos cortos (titulares, tuits)?
2. Una imagen generada por IA puede llevar un manifiesto C2PA perfectamente válido. ¿Qué acredita exactamente esa firma y por qué la ausencia de manifiesto no prueba origen humano?
3. La paráfrasis o el lavado con otro modelo eliminan la marca de agua sin conocer la clave. ¿Qué ataque análogo sufre C2PA (eliminar el manifiesto) y por qué la garantía de ambos esquemas es unidireccional?